# Embedding Pipeline: Batch Processing with Asyncio

This notebook processes ~200k strings through the embedding API using:
- **Asyncio**: 16 concurrent requests for high throughput
- **Batching**: Groups of 100 strings per API call
- **Progress Tracking**: tqdm for real-time monitoring
- **Storage**: Parquet format for fast reload without re-embedding

In [ ]:
import asyncio
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import time
from typing import List, Tuple, Dict, Any
import requests

# Patch asyncio for Jupyter compatibility
import nest_asyncio
nest_asyncio.apply()

# Import your existing API setup
from certara_secret import oauth2_token, envUrl, cert_verify
import os

os.environ["SSL_CERT_FILE"] = cert_verify
os.environ["REQUEST_CA_BUNDLE"] = cert_verify

## Step 1: API Wrapper Functions

Reusing synchronous API functions from `Asyncio.py` and wrapping them in async.

In [ ]:
# Install nest_asyncio for Jupyter compatibility
import subprocess
import sys

try:
    import nest_asyncio
except ImportError:
    print("Installing nest_asyncio...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nest_asyncio", "-q"])
    print("✓ nest_asyncio installed")

In [ ]:
# Synchronous API functions (from your Asyncio.py)
def get_api_service_url():
    return envUrl + "/layar/gpt/retreival/generateVectors"

def auth_token():
    return oauth2_token

def send_request(list_of_texts: List[str]) -> Dict[str, Any]:
    """Synchronous API call to generate embeddings."""
    api_service_url = get_api_service_url()
    api_key = auth_token()
    
    cmd = {
        "inputText": list_of_texts,
        "embedder": "multi-qa-MiniLM-L6-cos-v1"
    }
    
    try:
        response = requests.post(
            f"{api_service_url}",
            json=cmd,
            headers={
                "Authorization": f"Bearer {api_key}",
                "Accept": "application/json"
            },
            verify=cert_verify
        )
        
        llm_response = response.json()
        
        if response.status_code != 200:
            print(f"API Error: {llm_response}")
            raise Exception(f"Error: {llm_response.get('message', 'Unknown error')}")
        
        return llm_response
    
    except Exception as e:
        print(f"Request failed: {e}")
        raise

# Async wrapper with semaphore control and retry logic
async def async_send_request(
    texts: List[str],
    semaphore: asyncio.Semaphore,
    max_retries: int = 2
) -> Tuple[List[str], List[List[float]]]:
    """
    Async wrapper for API call with concurrency control and retry logic.
    
    Args:
        texts: List of strings to embed
        semaphore: Asyncio semaphore for concurrency control
        max_retries: Number of retry attempts for transient errors
    
    Returns:
        Tuple of (texts, embeddings) lists
    """
    async with semaphore:
        for attempt in range(max_retries):
            try:
                loop = asyncio.get_event_loop()
                response = await loop.run_in_executor(
                    None, send_request, texts
                )
                
                # Parse response
                data = response.get('data', [])
                texts_out = [item['text'] for item in data]
                embeddings_out = [item['embeddings'] for item in data]
                
                return texts_out, embeddings_out
            
            except Exception as e:
                if attempt < max_retries - 1:
                    await asyncio.sleep(2 ** attempt)  # Exponential backoff
                else:
                    print(f"Failed after {max_retries} retries: {e}")
                    raise

## Step 2: Batch Processor with Asyncio

Chunks input texts into batches and processes concurrently.

In [ ]:
async def process_batches(
    texts: List[str],
    batch_size: int = 100,
    max_concurrent: int = 16
) -> Tuple[List[str], List[List[float]]]:
    """
    Process texts in batches with concurrent API requests.
    
    Args:
        texts: List of all strings to embed
        batch_size: Number of strings per API call
        max_concurrent: Maximum concurrent API requests
    
    Returns:
        Tuple of (all_texts, all_embeddings) in original order
    """
    # Create batches
    batches = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
    
    # Create semaphore for concurrency control
    semaphore = asyncio.Semaphore(max_concurrent)
    
    # Create tasks for all batches
    tasks = [
        async_send_request(batch, semaphore)
        for batch in batches
    ]
    
    # Process all batches with progress bar
    results = []
    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Processing batches"):
        try:
            texts_batch, embeddings_batch = await coro
            results.append((texts_batch, embeddings_batch))
        except Exception as e:
            print(f"Batch processing error: {e}")
            continue
    
    # Flatten results while preserving order
    all_texts = []
    all_embeddings = []
    
    # Process in batch order (not completion order)
    for i, batch in enumerate(batches):
        for result_texts, result_embeddings in results:
            if result_texts == batch:
                all_texts.extend(result_texts)
                all_embeddings.extend(result_embeddings)
                break
    
    return all_texts, all_embeddings

## Step 3: Main Pipeline Function

Coordinates the full workflow: batching → async processing → DataFrame creation → Parquet storage.

In [ ]:
def embed_texts(
    df: pd.DataFrame,
    text_column: str = 'text',
    batch_size: int = 100,
    max_concurrent: int = 16,
    output_file: str = 'embeddings.parquet'
) -> pd.DataFrame:
    """
    Main pipeline: embed texts from DataFrame and save to Parquet.
    
    Args:
        df: Input DataFrame with text column
        text_column: Name of the column containing texts
        batch_size: Number of texts per API batch
        max_concurrent: Max concurrent API requests
        output_file: Path to save Parquet file
    
    Returns:
        DataFrame with columns ['text', 'embeddings']
    """
    start_time = time.time()
    
    # Extract texts
    texts = df[text_column].tolist()
    print(f"Processing {len(texts)} texts...")
    
    # Run async batch processor
    all_texts, all_embeddings = asyncio.run(
        process_batches(texts, batch_size, max_concurrent)
    )
    
    # Create output DataFrame
    df_output = pd.DataFrame({
        'text': all_texts,
        'embeddings': all_embeddings
    })
    
    # Save to Parquet
    df_output.to_parquet(output_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"\n✓ Processing complete in {elapsed:.1f}s")
    print(f"✓ Saved {len(df_output)} embeddings to {output_file}")
    
    return df_output

## Step 4: Test with Sample Data

Start small to verify the pipeline works before scaling to 200k strings.

In [ ]:
# Create sample DataFrame to test
sample_texts = [
    "the quick brown fox jumps over the lazy dog",
    "machine learning is transforming technology",
    "natural language processing enables AI",
    "embeddings capture semantic meaning",
    "clustering helps discover patterns in data",
]

df_sample = pd.DataFrame({'text': sample_texts})

# Run pipeline on sample
df_embeddings_sample = embed_texts(
    df_sample,
    text_column='text',
    batch_size=2,  # Small batch for testing
    max_concurrent=2,
    output_file='embeddings_sample.parquet'
)

# Display results
print("\nSample output:")
print(df_embeddings_sample.head())
print(f"\nEmbedding shape: {len(df_embeddings_sample.iloc[0]['embeddings'])}-dimensional")

## Step 5: Run on Full Dataset

Once verified with sample data, run on your full 200k strings.

In [ ]:
# Load your actual data
# Replace this with your actual data source
# df_full = pd.read_csv('your_data.csv')  # or your data loading method

# Uncomment to run on full dataset (200k strings):
# df_embeddings_full = embed_texts(
#     df_full,
#     text_column='text',  # Adjust column name if needed
#     batch_size=100,       # Strings per API call
#     max_concurrent=16,    # Concurrent requests
#     output_file='embeddings_full.parquet'
# )

print("To run on full dataset, uncomment the code above and load your data.")

## Step 6: Load Embeddings from Parquet

Reload previously computed embeddings without re-embedding.

In [ ]:
# Load previously computed embeddings
df_loaded = pd.read_parquet('embeddings_sample.parquet')

print(f"Loaded {len(df_loaded)} embeddings from Parquet")
print(df_loaded.head(2))

# For clustering/analysis, convert embeddings to numpy array
embeddings_array = np.array(df_loaded['embeddings'].tolist())
print(f"\nEmbeddings array shape: {embeddings_array.shape}")
print(f"Ready for clustering analysis!")

## Notes

- **Test first**: Run on sample data (5-10 texts) before scaling to 200k
- **Concurrency**: 16 concurrent requests balances throughput vs. API limits; adjust if you hit rate limits
- **Batch size**: 100 strings per batch; reduce if API calls fail
- **Parquet storage**: Fast reloading for clustering work; no need to re-embed
- **Retry logic**: Automatically retries transient errors (timeouts, etc.) with exponential backoff
- **Progress tracking**: tqdm shows real-time batch completion and ETA